In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import io

# The ASMC WIS2 collection endpoint
BASE_URL = "https://z9ppn1a4nj.execute-api.ap-southeast-1.amazonaws.com/v1/collections/jp1_afedr_750m_late/items"

# Define the timeframe from your graph
start_date = datetime(2025, 8, 2)
end_date = datetime(2025, 10, 30)

# Bounding box boundaries for your MLR matrix
sumatra_box = (95.0, -6.0, 106.0, 6.0)
kalimantan_box = (108.0, -5.0, 120.0, 5.0)

daily_records = []
current_date = start_date

print("Extracting and parsing text streams from ASMC API...\n")

while current_date <= end_date:
    date_str = current_date.strftime("%Y-%m-%d")
    
    # Query all text payloads generated on this calendar day
    params = {
        "datetime": f"{date_str}T00:00:00Z/{date_str}T23:59:59Z",
        "limit": 100 
    }
    
    s_count = 0
    k_count = 0
    
    try:
        response = requests.get(BASE_URL, params=params)
        payload = response.json()
        features = payload.get('features', [])
        
        for feature in features:
            # Extract the raw text file structure from the API content properties
            # If the API stores it directly in properties or an attachment link:
            properties = feature.get('properties', {})
            
            # The API returns files like the ones you uploaded.
            # We fetch the download block text directly.
            file_url = feature.get('links', [{}])[0].get('href') 
            if not file_url:
                continue
                
            file_res = requests.get(file_url)
            file_text = file_res.text
            
            # Read the file line by line just like your text snippet
            buf = io.StringIO(file_text)
            for line in buf:
                line = line.strip()
                # Skip comments and structural headers
                if not line or line.startswith('#') or line.startswith('[') or line.startswith('='):
                    continue
                
                # Split comma-separated metrics
                parts = line.split(',')
                if len(parts) >= 6:
                    try:
                        lat = float(parts[0].strip())
                        lon = float(parts[1].strip())
                        confidence = float(parts[5].strip())
                        
                        # --- MATCHING FILTER ---
                        # If you want to replicate the 'High' filter in the UI graph,
                        # uncomment the line below (High is usually >= 80% confidence)
                        # if confidence < 80: continue
                        
                        # Sort into geographic bins
                        if sumatra_box[0] <= lon <= sumatra_box[2] and sumatra_box[1] <= lat <= sumatra_box[3]:
                            s_count += 1
                        elif kalimantan_box[0] <= lon <= kalimantan_box[2] and kalimantan_box[1] <= lat <= kalimantan_box[3]:
                            k_count += 1
                    except ValueError:
                        continue # Skip malformed trailing metadata text
                        
        daily_records.append({
            "Date": date_str,
            "Sumatra_Hotspots": s_count,
            "Kalimantan_Hotspots": k_count
        })
        print(f"Parsed {date_str} -> Sumatra: {s_count} | Kalimantan: {k_count}")
        
    except Exception as e:
        print(f"Skipping {date_str} due to network error: {e}")
        
    current_date += timedelta(days=1)

# Generate CSV
df = pd.DataFrame(daily_records)
df.to_csv("malaysia_haze_regressors_2025.csv", index=False)
print("\nExtraction complete! Data saved to 'malaysia_haze_regressors_2025.csv'")

Extracting and parsing text streams from ASMC API...

Parsed 2025-08-02 -> Sumatra: 0 | Kalimantan: 0
Parsed 2025-08-03 -> Sumatra: 0 | Kalimantan: 0
Parsed 2025-08-04 -> Sumatra: 0 | Kalimantan: 0


KeyboardInterrupt: 

In [7]:
import time
import json
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Initialize browser
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Run in background
options.add_argument("--window-size=1920,1080")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 15)

print("Opening ASMC Hotspot Portal...")
driver.get("https://asmc.asean.org/asmc-hotspot/")

try:
    # 1. Click the VIIRS Tab (Directly on the main page DOM)
    print("Selecting VIIRS sensor stream...")
    viirs_tab = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), 'VIIRS') or contains(@href, 'viirs')]")))
    driver.execute_script("arguments[0].click();", viirs_tab)
    time.sleep(3)

    # 2. Select 'Last 90 days' duration dropdown
    print("Setting duration scope...")
    duration_dropdown = wait.until(EC.presence_of_element_located((By.ID, "duration")))
    for option in duration_dropdown.find_elements(By.TAG_NAME, "option"):
        if option.get_attribute("value") == "last90days":
            option.click()
            break

    # 3. Handle Target Region Checkboxes
    print("Configuring geographic boundaries...")
    regions = {
        "reg_sumatra": True,
        "reg_kalimantan": True,
        "reg_pen_msia": True,
        "reg_sab_swk": True,
        "reg_thailand": False,
        "reg_myanmar": False,
        "reg_cambodia": False,
        "reg_vietnam": False,
        "reg_laos": False,
        "reg_philippines": False
    }

    for reg_name, should_check in regions.items():
        try:
            checkbox = driver.find_element(By.NAME, reg_name)
            if checkbox.is_selected() != should_check:
                driver.execute_script("arguments[0].click();", checkbox)
        except Exception:
            pass

    # 4. Click the View Button
    print("Submitting query parameters...")
    view_button = driver.find_element(By.ID, "btnSubmit")
    driver.execute_script("arguments[0].click();", view_button)
    
    print("Waiting for chart metrics to populate...")
    time.sleep(6)  # Give the backend chart time to load data

    # 5. Extract data straight out of Highcharts instance or DOM Fallback
    print("Scraping raw matrix blocks out of window context...")
    highcharts_script = """
    try {
        if (typeof Highcharts !== 'undefined' && Highcharts.charts.length > 0) {
            // Find the active visible chart instance
            var activeChart = Highcharts.charts.find(c => c !== undefined);
            if (activeChart) {
                var data = {
                    categories: activeChart.xAxis[0].categories,
                    series: activeChart.series.map(s => ({ name: s.name, data: s.yData }))
                };
                return JSON.stringify(data);
            }
        }
        return JSON.stringify({error: "Highcharts array not found natively"});
    } catch(err) {
        return JSON.stringify({error: err.message});
    }
    """
    chart_json = driver.execute_script(highcharts_script)
    chart_data = json.loads(chart_json)

    if "error" in chart_data or not chart_data.get("series"):
        print(f"Primary method skipped: {chart_data.get('error', 'Empty series')}. Trying DOM table fallback extraction...")
        
        # Fallback Method: Force Highcharts to inject data table into the HTML DOM directly
        driver.execute_script("""
            var activeChart = Highcharts.charts.find(c => c !== undefined);
            if(activeChart) { activeChart.viewData(); }
        """)
        time.sleep(2)
        
        table = driver.find_element(By.CLASS_INDEX, "highcharts-data-table")
        df = pd.read_html(table.get_attribute('outerHTML'))[0]
        output_name = "asmc_scraped_hotspots.csv"
        df.to_csv(output_name, index=False)
        print(f"\nSuccess via Fallback Extraction! File generated: '{output_name}'")
        print(df.tail(10))

    else:
        # 6. Parse structured categories and metrics into table rows
        dates = chart_data['categories']
        df = pd.DataFrame({"Date": dates})
        
        for series in chart_data['series']:
            if series['data']:  # Confirm array has measurements
                df[series['name']] = series['data']
                
        output_name = "asmc_scraped_hotspots.csv"
        df.to_csv(output_name, index=False)
        print(f"\nSuccess via Engine Memory! File generated: '{output_name}'")
        print("\n--- DATA HEAD EXTRACT PREVIEW ---")
        print(df.tail(10))

except Exception as e:
    print(f"\nExtraction failed: {e}")

finally:
    driver.quit()
    print("Browser context shut down cleanly.")

Opening ASMC Hotspot Portal...
Selecting VIIRS sensor stream...

Extraction failed: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff66c877de5+14895]
	chromedriver!GetHandleVerifier [0x7ff66c877e50+14900]
	chromedriver!(No symbol) [0x7ff66c5dd5ad]
	chromedriver!(No symbol) [0x7ff66c637822]
	chromedriver!(No symbol) [0x7ff66c637b2c]
	chromedriver!(No symbol) [0x7ff66c687d17]
	chromedriver!(No symbol) [0x7ff66c68486f]
	chromedriver!(No symbol) [0x7ff66c629df8]
	chromedriver!(No symbol) [0x7ff66c62ace3]
	chromedriver!GetHandleVerifier [0x7ff66cb8cc49+3296f9]
	chromedriver!GetHandleVerifier [0x7ff66cb87375+323e25]
	chromedriver!GetHandleVerifier [0x7ff66cbabc82+348732]
	chromedriver!GetHandleVerifier [0x7ff66c896045+32af5]
	chromedriver!GetHandleVerifier [0x7ff66c89ecec+3b79c]
	chromedriver!GetHandleVerifier [0x7ff66c881bc4+1e674]
	chromedriver!GetHandleVerifier [0x7ff66c881d54+1e804]
	chromedriver!GetHandleVerifier [0x7ff66c8660e7+2b97]
	KERNEL32!BaseThreadInitThunk [0x7ffcc207